# 实验3：分治


---

## 题目1：数字旋转方阵程序设计

实验目的：

1. 掌握分治法的设计思想；
2. 掌握数字旋转方阵的具体实现过程；
3. 熟练掌握二维数组的使用方法；
4. 在掌握的基础上编程实现数字旋转方阵的实现过程。

实验内容：

给出一个初始数据，在此数据的基础上由外层向里层填写数据，完成一个数字旋转方阵，输出结果，输出时要求有文字说明。请任选一种语言编写程序实现上述算法，并分析其算法复杂度。


### 算法思路

从当前方阵的左上角开始，按上边向右、右边向下、下边向左、左边向上的顺序填写最外层，每填写一个格子，待填数字加1。通过调整各条边的起止位置，避免重复填写四个角。

外层填写完成后，将左上角的行、列下标各加1，边长减2，以当前待填数字作为内层的起始数字，递归填写内部方阵。

当边长为0时，直接返回；当边长为1时，将起始数字填入唯一的格子后返回。

### 伪代码
约定从左上角开始顺时针进行填充
```
Fill(A, row, col, size, start):

    如果 size == 0:
        return

    如果 size == 1:
        A[row][col] ← start
        return

    value ← start

    按上边、右边、下边、左边的顺序填写外层
    每填一格后，value 加1

    Fill(A, row + 1, col + 1, size - 2, value)
```


In [1]:
#include <stdio.h>
#include <stdlib.h>

In [2]:
// 顺时针填写当前方阵的外层，再递归填写内层。
// A的容量为100×100；调用时要求当前方阵位于数组边界内。
void fill(int A[100][100], int row, int col, int size, int start) {
    if (size <= 0) {
        return;
    }
    if (size == 1) {
        A[row][col] = start;
        return;
    }

    int value = start;
    int bottom = row + size - 1;
    int right = col + size - 1;

    // 上边包含两个角，其余各边避开已经填写的角。
    for (int j = col; j <= right; j++) {
        A[row][j] = value++;
    }
    for (int i = row + 1; i <= bottom; i++) {
        A[i][right] = value++;
    }
    for (int j = right - 1; j >= col; j--) {
        A[bottom][j] = value++;
    }
    for (int i = bottom - 1; i > row; i--) {
        A[i][col] = value++;
    }

    fill(A, row + 1, col + 1, size - 2, value);
}


In [3]:
{
    int A[100][100] = {0};
    int n = 7;
    int start = 5;

    fill(A, 0, 0, n, start);

    // 接下来输出 A 中前 n 行、前 n 列的元素。
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            printf("%4d", A[i][j]);
        }
        printf("\n");
    }
}

   5   6   7   8   9  10  11
  28  29  30  31  32  33  12
  27  44  45  46  47  34  13
  26  43  52  53  48  35  14
  25  42  51  50  49  36  15
  24  41  40  39  38  37  16
  23  22  21  20  19  18  17


### 复杂度分析

以向一个格子写入数字作为基本操作。对于边长为 n 的方阵，每个格子恰好填写一次，总填写次数为 n²，因此时间复杂度为 O(n²)。输出整个方阵也需要访问 n² 个元素，包含输出时的整体时间复杂度仍为 O(n²)。

每次递归将边长减2，递归深度约为 n/2。每层调用只使用固定数量的参数和局部变量，因此按通常的递归调用栈模型，额外空间复杂度为 O(n)。

保存 n×n 方阵的结果需要 O(n²) 的存储空间。

---

## 题目2：排序中分治法的程序设计

实验目的：

1. 掌握归并排序和快速排序的划分方法；
2. 掌握归并排序和快速排序的具体分治策略；
3. 在掌握的基础上编程两种排序方法的实现过程。

实验内容：

给出一个初始序列，分别用归并排序和快速排序两种分治法将所给序列变换为有序序列，输出结果，输出时要求有文字说明。请任选一种语言编写程序实现上述算法，并分析其算法复杂度。


### 归并排序

#### 算法思路
归并排序采用分治法，将一个较大的排序问题分解为两个规模较小的排序问题，再通过合并得到完整的有序序列。本实验按从小到大的顺序排序。

1. 分解：用 left 和 right 表示当前待排序区间的左右端点，两个端点均包含在区间内。计算中间下标 mid，将当前区间划分为 left～mid 和 mid+1～right 两部分。
2. 求解：递归地对左右两部分进行归并排序。当 left >= right 时，当前区间至多只有一个元素，已经有序，直接返回。
3. 合并：左右两部分分别有序后，使用两个下标指向各部分尚未取出的第一个元素，比较两者，将较小的元素依次写入临时数组。若两者相等，先取左半部分的元素。
4. 当某一部分的元素全部取完后，将另一部分剩余的元素按原顺序追加到临时数组中。
5. 将临时数组中的有序结果复制回原数组的 left～right 区间，完成当前区间的排序。

#### 伪代码实现
约定：
- 数组下标从0开始。
- left、mid、right 所表示的区间均包含端点。
- temp 是容量足以保存当前区间全部元素的临时数组。

```text
MergeSort(A, left, right):
    如果 left >= right:
        返回

    mid ← left + (right - left) / 2 的整数商

    MergeSort(A, left, mid)
    MergeSort(A, mid + 1, right)

    Merge(A, left, mid, right)


Merge(A, left, mid, right):
    创建临时数组 temp，容量为 right - left + 1

    i ← left
    j ← mid + 1
    k ← 0

    // 两边都有剩余元素时，比较并取出较小的元素
    当 i <= mid 且 j <= right:
        如果 A[i] <= A[j]:
            temp[k] ← A[i]
            i ← i + 1
        否则:
            temp[k] ← A[j]
            j ← j + 1
        k ← k + 1

    // 追加左半部分剩余的元素
    当 i <= mid:
        temp[k] ← A[i]
        i ← i + 1
        k ← k + 1

    // 追加右半部分剩余的元素
    当 j <= right:
        temp[k] ← A[j]
        j ← j + 1
        k ← k + 1

    // 将合并结果复制回原数组的当前区间
    对 t 从 0 到 k - 1（包含端点）:
        A[left + t] ← temp[t]


首次调用:
    MergeSort(A, 0, n - 1)
```

#### 代码实现

In [4]:
void merge(int arr[], int left, int mid, int right);

// 归并排序：对闭区间 [left, right] 进行排序。
void merge_sort(int arr[], int left, int right) {
    // 空区间或只有一个元素时，无需排序。
    if (left >= right) {
        return;
    }

    int mid = left + (right - left) / 2;

    merge_sort(arr, left, mid);
    merge_sort(arr, mid + 1, right);

    // 左右两部分分别有序后，再进行合并。
    merge(arr, left, mid, right);
}

// 合并两个有序闭区间：left～mid 和 mid+1～right。
// 调用时要求 0 <= left <= mid < right，且 right 未超出数组边界。
void merge(int arr[], int left, int mid, int right) {
    // 动态申请临时空间，使用指针偏移访问元素。
    size_t count = (size_t)(right - left) + 1;
    int *tmp = malloc(count * sizeof *tmp);
    if (tmp == NULL) {
        fprintf(stderr, "归并排序：临时内存分配失败。\n");
        exit(EXIT_FAILURE);
    }

    int i = left;
    int j = mid + 1;
    int k = 0;

    // 两边都有元素时，取出较小的元素；相等时先取左边。
    while (i <= mid && j <= right) {
        if (arr[i] <= arr[j]) {
            *(tmp + k) = arr[i];
            i++;
        } else {
            *(tmp + k) = arr[j];
            j++;
        }
        k++;
    }

    // 追加左半部分剩余元素。
    while (i <= mid) {
        *(tmp + k) = arr[i];
        i++;
        k++;
    }

    // 追加右半部分剩余元素。
    while (j <= right) {
        *(tmp + k) = arr[j];
        j++;
        k++;
    }

    // 临时空间从0开始，原数组的目标区间从left开始。
    for (int p = 0; p < k; p++) {
        arr[left + p] = *(tmp + p);
    }

    // 复制完成后释放内存，tmp仍保存着分配时的起始地址。
    free(tmp);
}


### 快速排序

#### 算法思路
快速排序采用分治法。本实验按从小到大的顺序排序，选取当前区间的最后一个元素作为基准。

1. 划分：从左向右扫描除基准外的元素，维护一个“不大于基准”的区域。遇到不大于基准的元素时，将它交换到该区域的下一个位置，并扩大该区域。
2. 扫描结束后，将基准与该区域后面的第一个元素交换，使基准到达最终排序位置。此时，基准左侧的元素均不大于基准，右侧的元素均大于基准，但两侧内部不一定有序。
3. 递归地对基准左侧和右侧的区间进行快速排序。基准已经定位，不再参与后续递归。
4. 当待排序区间至多只有一个元素时，直接返回。左右两侧排序完成后，整个区间即为有序，无需额外合并。

#### 伪代码实现
约定：
- 数组下标从0开始。
- left 和 right 表示当前区间的左右端点，包含两端。
- Partition 返回基准划分后的最终下标。
- 循环起点大于终点时，不执行循环。

```text
QuickSort(A, left, right):
    如果 left >= right:
        返回

    p ← Partition(A, left, right)

    QuickSort(A, left, p - 1)
    QuickSort(A, p + 1, right)


Partition(A, left, right):
    pivot ← A[right]
    i ← left

    // i 指向下一个“不大于基准”的元素应放入的位置
    对 j 从 left 到 right - 1（包含端点）:
        如果 A[j] <= pivot:
            交换 A[i] 和 A[j]
            i ← i + 1

    // 将基准放到最终位置
    交换 A[i] 和 A[right]

    返回 i


首次调用:
    QuickSort(A, 0, n - 1)
```

#### 代码实现

In [5]:
int partition(int arr[], int left, int right){
    int pivot = arr[right]; // 选择最后一个元素作为基准
    int i = left;

    for(int j = left; j < right; j++){
        if(arr[j] < pivot){
            int tmp = arr[i];
            arr[i] = arr[j];
            arr[j] = tmp;
            i++;
        }
    }
    int tmp = arr[i];
    arr[i] = arr[right];
    arr[right] = tmp;

    return i; // 返回基准元素的最终位置
}

// 快速排序：对闭区间 [left, right] 进行排序。  
void quick_sort(int arr[], int left, int right){
    if(left >= right){
        return;
    }

    int p = partition(arr, left, right);

    quick_sort(arr, left, p - 1);
    quick_sort(arr, p + 1, right);
}

In [6]:
// 两种排序使用相同的输入和预期结果，分别检查各自的输出。
void print_sort_array(const int arr[], int n) {
    printf("[");
    for (int i = 0; i < n; i++) {
        printf("%s%d", i == 0 ? "" : ", ", arr[i]);
    }
    printf("]");
}

// sort_func 可以接收 quick_sort 或 merge_sort。
int run_sort_cases(const char *name,
                   void (*sort_func)(int[], int, int)) {
    struct SortCase {
        const char *name;
        int input[8];
        int expected[8];
        int n;
    };

    const struct SortCase cases[] = {
        {"普通乱序", {9, 4, 7, 2, 6}, {2, 4, 6, 7, 9}, 5},
        {"负数和重复值", {3, -1, 3, 0, -5, 2}, {-5, -1, 0, 2, 3, 3}, 6},
        {"已经有序", {1, 2, 3, 4, 5}, {1, 2, 3, 4, 5}, 5},
        {"逆序", {5, 4, 3, 2, 1}, {1, 2, 3, 4, 5}, 5},
        {"全部相等", {7, 7, 7, 7}, {7, 7, 7, 7}, 4},
        {"单个元素", {42}, {42}, 1},
        {"空区间", {0}, {0}, 0}
    };
    int total = (int)(sizeof cases / sizeof cases[0]);
    int passed = 0;

    printf("\n=== %s测试 ===\n", name);
    for (int c = 0; c < total; c++) {
        int arr[8];
        int n = cases[c].n;

        // 每次复制原始数据，避免两种排序相互影响。
        for (int i = 0; i < n; i++) {
            arr[i] = cases[c].input[i];
        }

        printf("%s\n  排序前：", cases[c].name);
        print_sort_array(arr, n);

        // n为0时，区间为[0, -1]，排序函数应直接返回。
        sort_func(arr, 0, n - 1);

        int correct = 1;
        for (int i = 0; i < n; i++) {
            if (arr[i] != cases[c].expected[i]) {
                correct = 0;
                break;
            }
        }

        printf("\n  排序后：");
        print_sort_array(arr, n);
        printf("\n  预期值：");
        print_sort_array(cases[c].expected, n);
        printf("\n  检查结果：%s\n", correct ? "通过" : "失败");
        passed += correct;
    }

    printf("%s：%d/%d 组通过。\n", name, passed, total);
    return total - passed;
}

int test_quick_sort(void) {
    return run_sort_cases("快速排序", quick_sort);
}

int test_merge_sort(void) {
    return run_sort_cases("归并排序", merge_sort);
}

// 统一入口：返回0表示全部通过，非0表示存在失败。
int main(void) {
    int failed = 0;
    failed += test_quick_sort();
    failed += test_merge_sort();
    printf("\n两种排序测试完成，共%d组失败。\n", failed);
    return failed == 0 ? 0 : 1;
}

// Jupyter中定义main后需要显式调用，执行本单元格即可运行测试。
// 导出为独立C程序时，删除下面这段调用即可。
{
    main();
}



=== 快速排序测试 ===
普通乱序
  排序前：[9, 4, 7, 2, 6]
  排序后：[2, 4, 6, 7, 9]
  预期值：[2, 4, 6, 7, 9]
  检查结果：通过
负数和重复值
  排序前：[3, -1, 3, 0, -5, 2]
  排序后：[-5, -1, 0, 2, 3, 3]
  预期值：[-5, -1, 0, 2, 3, 3]
  检查结果：通过
已经有序
  排序前：[1, 2, 3, 4, 5]
  排序后：[1, 2, 3, 4, 5]
  预期值：[1, 2, 3, 4, 5]
  检查结果：通过
逆序
  排序前：[5, 4, 3, 2, 1]
  排序后：[1, 2, 3, 4, 5]
  预期值：[1, 2, 3, 4, 5]
  检查结果：通过
全部相等
  排序前：[7, 7, 7, 7]
  排序后：[7, 7, 7, 7]
  预期值：[7, 7, 7, 7]
  检查结果：通过
单个元素
  排序前：[42]
  排序后：[42]
  预期值：[42]
  检查结果：通过
空区间
  排序前：[]
  排序后：[]
  预期值：[]
  检查结果：通过
快速排序：7/7 组通过。

=== 归并排序测试 ===
普通乱序
  排序前：[9, 4, 7, 2, 6]
  排序后：[2, 4, 6, 7, 9]
  预期值：[2, 4, 6, 7, 9]
  检查结果：通过
负数和重复值
  排序前：[3, -1, 3, 0, -5, 2]
  排序后：[-5, -1, 0, 2, 3, 3]
  预期值：[-5, -1, 0, 2, 3, 3]
  检查结果：通过
已经有序
  排序前：[1, 2, 3, 4, 5]
  排序后：[1, 2, 3, 4, 5]
  预期值：[1, 2, 3, 4, 5]
  检查结果：通过
逆序
  排序前：[5, 4, 3, 2, 1]
  排序后：[1, 2, 3, 4, 5]
  预期值：[1, 2, 3, 4, 5]
  检查结果：通过
全部相等
  排序前：[7, 7, 7, 7]
  排序后：[7, 7, 7, 7]
  预期值：[7, 7, 7, 7]
  检查结果：通过
单个元素
  排序前：[42]
  排序后：[42]
  预期值：[42]
  检查结

### 复杂度分析

设待排序序列的元素个数为 n。以下分析针对排序函数本身，不包含测试程序的数据复制、结果检查与输出。

#### 1. 归并排序

**时间复杂度**

归并排序每次将当前区间划分为规模近似相等的两部分，递归排序后再合并。对于包含 m 个元素的区间，合并时将全部元素写入临时空间，再复制回原数组，工作量为 O(m)。即使输入已经有序，本实现也仍然执行合并和复制。

当 n 为2的幂时，递推关系可写为：

$$
T(n) = 2T(n/2) + O(n)
$$

递归拆分约有 log₂n 层，每层合并处理的元素总量为 O(n)，因此最好、平均和最坏时间复杂度均为 O(n log n)。n 不是2的幂时，两部分的规模至多相差1，复杂度结论相同。

**额外空间复杂度**

当前实现中，merge 使用 malloc 为当前合并区间申请临时空间，并在复制结束后调用 free 释放。最大的一次合并需要保存 n 个整数，因此临时空间的峰值为 O(n)。

左右递归调用依次执行，且当前层的临时空间在两侧递归返回后才申请；各次合并使用的临时空间不会全部同时保留，不能把整个执行过程中的申请总量当成峰值空间。递归调用栈深度为 O(log n)，因此总额外空间为 O(n) + O(log n) = O(n)。

#### 2. 快速排序

**一次划分的工作量**

当前代码选择区间最后一个元素作为基准，通过 arr[j] < pivot 判断元素是否进入左侧区域。划分后，基准左侧元素小于基准，右侧元素大于或等于基准。

对包含 m 个元素的区间，partition 扫描其余 m-1 个元素，每次比较和交换使用常数时间，因此一次划分的时间复杂度为 O(m)。

**最好时间复杂度**

当每次划分产生的左右区间规模近似相等时，递归深度为 O(log n)，每层划分的总工作量为 O(n)，因此最好时间复杂度为 O(n log n)。

**平均时间复杂度**

对于元素互异、各排列等可能的随机输入，平均时间复杂度为 O(n log n)。这一结论具有输入分布前提；当前代码固定选择末尾元素作为基准，并不保证对任意输入都能得到均衡划分。

**最坏时间复杂度**

当每次基准都是当前区间的最小值或最大值时，两侧规模分别为0和 m-1，问题规模每次只减少1，递推关系为：

$$
T(n) = T(n-1) + O(n)
$$

基准比较次数为：

$$
(n-1) + (n-2) + \cdots + 1 = \frac{n(n-1)}{2}
$$

因此最坏时间复杂度为 O(n²)。对于本实现，升序、降序以及全部元素相等的序列都会形成这样的退化划分。全部相等时，由于使用严格小于比较，没有元素进入左侧区域，基准每次都落在当前区间的最左端。

**额外空间复杂度**

partition 只使用固定数量的辅助变量，单次划分的额外空间为 O(1)。整个快速排序还需要计算递归调用栈的空间：划分均衡时为 O(log n)；在上述随机输入假设下，平均为 O(log n)；最坏情况下递归深度为 O(n)，额外空间也为 O(n)。

本分析按普通递归调用栈模型计算，不依赖编译器的尾调用优化。

#### 3. 对比

| 算法 | 最好时间 | 平均时间 | 最坏时间 | 额外空间（不含输入数组） |
|---|---|---|---|---|
| 归并排序 | O(n log n) | O(n log n) | O(n log n) | O(n) |
| 快速排序 | O(n log n) | O(n log n)，基于上述随机输入假设 | O(n²) | 均衡时 O(log n)，上述假设下平均 O(log n)，最坏 O(n) |

归并排序按位置近似均分，时间复杂度不依赖划分是否均衡，但需要线性临时空间。快速排序直接在原数组中划分，不需要线性临时数组，其时间与调用栈空间取决于划分结果。

前面的功能测试用于检查排序结果是否正确，未进行性能计时，不能据此判断哪一种排序实际运行得更快。


---

## 题目3：汉诺塔问题的程序设计

实验目的：

1. 掌握递归的有关概念；
2. 掌握汉诺塔问题的具体求解过程；
3. 在掌握的基础上编程实现汉诺塔的具体实现过程。

实验内容：

在 A 上有按大小排序好的 n 个金碟，借助 B 的帮助，将 A 上的碟子移动到 C 上，在移动的过程中要严格按照大小顺序，不能将碟子放在比它小的上面，输出结果，输出时要求有文字说明。请任选一种语言编写程序实现上述算法，并分析其算法复杂度。


### 伪代码实现

约定：
- 盘号从1开始，编号越大，盘越大。
- source、auxiliary、target 分别表示本次任务的起始柱、辅助柱和目标柱。
- count 以引用方式传递，所有递归调用共享同一个计数器；对应 C 实现中的指针参数。
- 每实际移动一个盘，计数器加1；首次调用前将计数器初始化为0。

```text
Hanoi(n, source, auxiliary, target, 引用 count):
    如果 n <= 0:
        返回

    如果 n == 1:
        count ← count + 1
        输出“第 count 步：将1号盘从 source 移动到 target”
        返回

    // 第一阶段：将上面的 n-1 个盘从起始柱移到辅助柱，借助目标柱
    Hanoi(n - 1, source, target, auxiliary, 引用 count)

    // 第二阶段：将最大的 n 号盘从起始柱移到目标柱
    count ← count + 1
    输出“第 count 步：将 n 号盘从 source 移动到 target”

    // 第三阶段：将 n-1 个盘从辅助柱移到目标柱，借助原起始柱
    Hanoi(n - 1, auxiliary, source, target, 引用 count)


主过程:
    设置盘数 n
    count ← 0

    如果 n < 0:
        输出“盘子数量不能为负数”
    否则:
        输出“将 n 个盘从 A 移到 C，借助 B”
        Hanoi(n, 'A', 'B', 'C', 引用 count)
        输出“移动完成，总步数：count”
```


In [7]:
/**
 * 汉诺塔：将n个盘从source移到target，借助auxiliary。
 * count指向本次运行共用的步数计数器，首次调用前应初始化为0。
 */
void hanoi(int n, char source, char auxiliary, char target,
           unsigned long long *count) {
    // 没有盘子时不移动，也不增加计数。
    if (n <= 0) {
        return;
    }

    if (n == 1) {
        (*count)++;
        printf("第%llu步：将1号盘从 %c 移动到 %c\n",
               *count, source, target);
        return;
    }

    // 第一阶段：将上面的n-1个盘从源柱移到辅助柱。
    hanoi(n - 1, source, target, auxiliary, count);

    // 第二阶段：移动第n号盘；一次实际移动对应一次计数。
    (*count)++;
    printf("第%llu步：将%d号盘从 %c 移动到 %c\n",
           *count, n, source, target);

    // 第三阶段：将n-1个盘从辅助柱移到目标柱。
    hanoi(n - 1, auxiliary, source, target, count);
}


In [8]:
{
    int n = 5;
    // 每次运行从0开始，所有递归调用通过地址共享这个计数器。
    unsigned long long count = 0;

    if (n < 0) {
        printf("盘子数量不能为负数。\n");
    } else {
        printf("汉诺塔：将%d个盘从 A 移到 C，借助 B。\n", n);
        hanoi(n, 'A', 'B', 'C', &count);
        printf("移动完成，总步数：%llu。\n", count);
    }
}


汉诺塔：将5个盘从 A 移到 C，借助 B。
第1步：将1号盘从 A 移动到 C
第2步：将2号盘从 A 移动到 B
第3步：将1号盘从 C 移动到 B
第4步：将3号盘从 A 移动到 C
第5步：将1号盘从 B 移动到 A
第6步：将2号盘从 B 移动到 C
第7步：将1号盘从 A 移动到 C
第8步：将4号盘从 A 移动到 B
第9步：将1号盘从 C 移动到 B
第10步：将2号盘从 C 移动到 A
第11步：将1号盘从 B 移动到 A
第12步：将3号盘从 C 移动到 B
第13步：将1号盘从 A 移动到 C
第14步：将2号盘从 A 移动到 B
第15步：将1号盘从 C 移动到 B
第16步：将5号盘从 A 移动到 C
第17步：将1号盘从 B 移动到 A
第18步：将2号盘从 B 移动到 C
第19步：将1号盘从 A 移动到 C
第20步：将3号盘从 B 移动到 A
第21步：将1号盘从 C 移动到 B
第22步：将2号盘从 C 移动到 A
第23步：将1号盘从 B 移动到 A
第24步：将4号盘从 B 移动到 C
第25步：将1号盘从 A 移动到 C
第26步：将2号盘从 A 移动到 B
第27步：将1号盘从 C 移动到 B
第28步：将3号盘从 A 移动到 C
第29步：将1号盘从 B 移动到 A
第30步：将2号盘从 B 移动到 C
第31步：将1号盘从 A 移动到 C
移动完成，总步数：31。


### 复杂度分析

设移动 n 个盘需要 M(n) 次移动。算法先移动 n-1 个盘，再移动最大的盘，最后再次移动 n-1 个盘，因此：

M(1) = 1
M(n) = 2M(n-1) + 1

由此可得 M(n) = 2^n - 1。按每次移动的计数与输出为常数时间计算，时间复杂度为 O(2^n)。

每条递归路径上的盘数依次从 n 减少到1，最大调用层数为 n。两个递归分支先后执行，第一个分支返回后才开始第二个分支，因此不会同时保留两个分支的全部调用。每层调用使用固定数量的参数和局部信息，按普通递归调用栈模型，额外空间复杂度为 O(n)。

计数器由所有递归调用共享，只占固定空间，不改变上述空间复杂度。

---

## 题目4：查找中减治法的程序设计

实验目的：

1. 掌握减治法的设计思想；
2. 掌握折半查找和二叉查找的思想及实现过程；
3. 在掌握的基础上编程实现两种查找方法的具体实现过程。

实验内容：

给出一个序列及要查找的值，分别用两种查找方法实现，输出结果，输出时要求有文字说明。

请任选一种语言编写程序实现上述算法，并分析其算法复杂度。


### 二分查找

#### 算法思路

二分查找（折半查找）采用减治法，要求输入数组已按升序排列。

1. 用 left 和 right 表示当前查找区间的左右端点，区间包含两端。初始时 left = 0，right = n - 1。
2. 当 left <= right 时，计算中间下标 mid = left + (right - left) / 2，其中除法取整数商。
3. 将中间元素 A[mid] 与目标值 key 比较：相等时返回 mid；若 key > A[mid]，令 left = mid + 1，只保留右半部分；否则令 right = mid - 1，只保留左半部分。
4. 每轮都排除已经检查过的中间元素及不可能包含目标的一侧，将问题规模缩小约一半，继续查找。
5. 当 left > right 时，候选区间为空，返回 -1，表示目标不存在。left == right 时仍有一个元素需要检查。若数组包含重复值，当前实现返回其中一个匹配位置，不保证为首次或末次出现的位置。

#### 伪代码实现
```text
BinarySearch(A, n, key):
    left ← 0
    right ← n - 1
当 left <= right:
    mid ← left + (right - left) / 2 的整数商

    如果 A[mid] == key:
        返回 mid
    否则如果 key > A[mid]:
        left ← mid + 1
    否则:
        right ← mid - 1

返回 -1
```

#### 代码实现

In [9]:
// 在升序数组中查找key，找到返回下标，否则返回-1。
int binary_search(int A[], int n, int key) {
    int left = 0;
    int right = n - 1;

    while (left <= right) {
        int mid = left + (right - left) / 2;

        if (A[mid] == key) {
            return mid;
        } else if (A[mid] < key) {
            left = mid + 1;
        } else {
            right = mid - 1;
        }
    }

    return -1;
}

In [10]:
// 检查一次二分查找的结果，返回1表示通过，0表示失败。
int check_binary_search(const char *name, int A[], int n,
                        int key, int expected) {
    int actual = binary_search(A, n, key);

    printf("%s：查找%d，预期下标%d，实际下标%d，%s\n",
           name, key, expected, actual,
           actual == expected ? "通过" : "失败");

    return actual == expected;
}

void test_binary_search(void) {
    int A[] = {2, 5, 8, 11, 13, 17, 20};
    int n = sizeof(A) / sizeof(A[0]);

    int single[] = {8};
    int passed = 0;
    int total = 0;

    passed += check_binary_search("找到中间元素", A, n, 11, 3);
    total++;

    passed += check_binary_search("找到首元素", A, n, 2, 0);
    total++;

    passed += check_binary_search("找到末元素", A, n, 20, 6);
    total++;

    passed += check_binary_search("目标位于两个元素之间", A, n, 14, -1);
    total++;

    passed += check_binary_search("目标小于全部元素", A, n, 1, -1);
    total++;

    passed += check_binary_search("目标大于全部元素", A, n, 21, -1);
    total++;

    passed += check_binary_search("单个元素且找到", single, 1, 8, 0);
    total++;

    passed += check_binary_search("单个元素但未找到", single, 1, 9, -1);
    total++;

    // n为0时初始范围为[0, -1]，函数应直接返回，不访问数组。
    passed += check_binary_search("空区间", single, 0, 8, -1);
    total++;

    printf("\n二分查找测试：%d/%d组通过。\n", passed, total);
}

// 在Jupyter中显式调用测试函数。
{
    test_binary_search();
}

找到中间元素：查找11，预期下标3，实际下标3，通过
找到首元素：查找2，预期下标0，实际下标0，通过
找到末元素：查找20，预期下标6，实际下标6，通过
目标位于两个元素之间：查找14，预期下标-1，实际下标-1，通过
目标小于全部元素：查找1，预期下标-1，实际下标-1，通过
目标大于全部元素：查找21，预期下标-1，实际下标-1，通过
单个元素且找到：查找8，预期下标0，实际下标0，通过
单个元素但未找到：查找9，预期下标-1，实际下标-1，通过
空区间：查找8，预期下标-1，实际下标-1，通过

二分查找测试：9/9组通过。


### 二叉查找

#### 算法思路
将任意序列进行树化，构建二叉搜索树，并根据二叉搜索树的性质（左小右大）进行对应的查找。
#### 伪代码实现
```text
BSTSearch(root, key):
    current ← root

    当 current 不为空:
        如果 key == current.value:
            返回 current
        否则如果 key < current.value:
            current ← current.left
        否则:
            current ← current.right

    返回 空

BSTInsert(root, value):
    如果 root 为空:
        创建节点 node
        node.value ← value
        node.left ← 空
        node.right ← 空
        返回 node

    如果 value < root.value:
        root.left ← BSTInsert(root.left, value)
    否则如果 value > root.value:
        root.right ← BSTInsert(root.right, value)

    返回 root
```


#### 代码实现

In [11]:
// 二叉搜索树节点
typedef struct BSTNode {
    int value;               // 当前节点的值
    struct BSTNode *left;    // 左孩子
    struct BSTNode *right;   // 右孩子
} BSTNode;

// 找到时返回节点指针，未找到时返回NULL。
BSTNode *bst_search(BSTNode *root, int key) {
    BSTNode *current = root;

    while (current != NULL) {
        if (current->value == key) {
            return current;
        } else if (current->value > key) {
            current = current->left;
        } else {
            current = current->right;
        }
    }

    return NULL;
}

In [12]:
// 插入value，返回插入后的子树根节点；忽略重复值。
BSTNode *bst_insert(BSTNode *root, int value) {
    if (root == NULL) {
        BSTNode *node = malloc(sizeof *node);

        if (node == NULL) {
            fprintf(stderr, "创建二叉搜索树节点失败。\n");
            exit(EXIT_FAILURE);
        }

        node->value = value;
        node->left = NULL;
        node->right = NULL;

        return node;
    }

    if (value < root->value) {
        root->left = bst_insert(root->left, value);
    } else if (value > root->value) {
        root->right = bst_insert(root->right, value);
    }

    return root;
}

In [13]:
// 后序释放：先释放左右子树，再释放当前节点。
void bst_free(BSTNode *root) {
    if (root == NULL) {
        return;
    }
    bst_free(root->left);
    bst_free(root->right);
    free(root);
}

// 统计实际节点数，用于检查重复值没有创建新节点。
int bst_node_count(BSTNode *root) {
    if (root == NULL) {
        return 0;
    }
    return 1 + bst_node_count(root->left) + bst_node_count(root->right);
}

// 一个场景：按输入顺序建树，然后检查多次查找，最后释放树。
int run_bst_case(const char *name, const int values[], int n,
                 const int keys[], const int expected_found[],
                 int query_count, int expected_nodes) {
    BSTNode *root = NULL;
    int failed = 0;

    printf("\n=== %s ===\n输入序列：", name);
    for (int i = 0; i < n; i++) {
        printf("%d ", values[i]);
        root = bst_insert(root, values[i]);
    }
    if (n == 0) {
        printf("空序列");
    }
    printf("\n");

    int actual_nodes = bst_node_count(root);
    int count_ok = actual_nodes == expected_nodes;
    printf("节点数：预期%d，实际%d，%s\n",
           expected_nodes, actual_nodes, count_ok ? "通过" : "失败");
    failed += !count_ok;

    for (int i = 0; i < query_count; i++) {
        BSTNode *result = bst_search(root, keys[i]);
        // 找到时还要确认返回节点中的值等于目标。
        int correct = expected_found[i]
            ? (result != NULL && result->value == keys[i])
            : (result == NULL);

        printf("查找%d：预期%s，实际%s，%s\n",
               keys[i], expected_found[i] ? "找到" : "未找到",
               result != NULL ? "找到" : "未找到",
               correct ? "通过" : "失败");
        failed += !correct;
    }

    bst_free(root);
    root = NULL;
    return failed;
}

void test_bst(void) {
    int failed = 0;

    // 包含根节点、内部节点、叶节点及不存在的值；重复插入6。
    int values1[] = {8, 3, 12, 6, 1, 10, 15, 4, 6};
    int keys1[] = {8, 6, 4, 11, -1, 20};
    int expected1[] = {1, 1, 1, 0, 0, 0};
    failed += run_bst_case("普通建树、重复值与查找",
                          values1, 9, keys1, expected1, 6, 8);

    // 升序插入会形成向右延伸的树，查找仍应正确。
    int values2[] = {1, 2, 3, 4, 5};
    int keys2[] = {5, 0};
    int expected2[] = {1, 0};
    failed += run_bst_case("升序输入",
                          values2, 5, keys2, expected2, 2, 5);

    int values3[] = {-3};
    int keys3[] = {-3, 8};
    int expected3[] = {1, 0};
    failed += run_bst_case("单节点与负数",
                          values3, 1, keys3, expected3, 2, 1);

    // n为0时不会读取values，根节点始终为NULL。
    int keys4[] = {8};
    int expected4[] = {0};
    failed += run_bst_case("空树",
                          NULL, 0, keys4, expected4, 1, 0);

    printf("\n二叉搜索树测试：15项检查，%d项失败。\n", failed);
}

// 在Jupyter中显式调用。
{
    test_bst();
}



=== 普通建树、重复值与查找 ===
输入序列：8 3 12 6 1 10 15 4 6 
节点数：预期8，实际8，通过
查找8：预期找到，实际找到，通过
查找6：预期找到，实际找到，通过
查找4：预期找到，实际找到，通过
查找11：预期未找到，实际未找到，通过
查找-1：预期未找到，实际未找到，通过
查找20：预期未找到，实际未找到，通过

=== 升序输入 ===
输入序列：1 2 3 4 5 
节点数：预期5，实际5，通过
查找5：预期找到，实际找到，通过
查找0：预期未找到，实际未找到，通过

=== 单节点与负数 ===
输入序列：-3 
节点数：预期1，实际1，通过
查找-3：预期找到，实际找到，通过
查找8：预期未找到，实际未找到，通过

=== 空树 ===
输入序列：空序列
节点数：预期0，实际0，通过
查找8：预期未找到，实际未找到，通过

二叉搜索树测试：15项检查，0项失败。


### 复杂度分析

- **折半查找**：要求数组已升序排列。每轮将候选范围缩小约一半，最好时间复杂度为 O(1)，最坏为 O(log n)。当前循环实现的额外空间为 O(1)，不包含预先排序的成本。
- **二叉搜索树查找**：查找沿一条路径进行，最好时间为 O(1)，最坏时间为 O(h)，其中 h 为树高。树均衡时为 O(log n)，退化成链时为 O(n)。当前循环查找的额外空间为 O(1)。
- **建树**：逐个插入 n 个不同元素，若插入过程中树始终均衡，时间为 O(n log n)；最坏退化成链时为 O(n²)。递归插入的调用栈空间在均衡时为 O(log n)，最坏为 O(n)，另需 O(n) 空间存储节点。

当前二叉搜索树没有自动平衡操作，因此不保证对数级查找。分析“建树后查找”的完整流程时，还需计入建树成本。


---

## 题目5：排序中减治法的程序设计

实验目的：

1. 掌握堆的有关概念；
2. 掌握堆排序的基本思想和其算法的实现过程；
3. 熟练掌握筛选算法的实现过程；
4. 在掌握的基础上编程实现堆排序的具体实现过程。

实验内容：

给出一个记录序列，用堆排序的方法将其进行升序排列，输出结果，输出时要求有文字说明。

请任选一种语言编写程序实现上述算法，并分析其算法复杂度。


### 算法思路


1. 用数组表示完全二叉树，下标从0开始。节点 i 的左右孩子下标分别为 2*i+1 和 2*i+2，孩子下标必须小于当前堆的元素个数。
2. 建立大顶堆，使每个父节点都不小于其孩子，最大值位于堆顶。左右孩子之间不要求有序。
3. 从最后一个非叶子节点 n/2-1 开始，自底向上执行下沉调整。调整一个节点时，其左右子树应已分别满足大顶堆性质。
4. 将堆顶与当前堆的末尾元素交换，最大值便到达最终位置；堆的规模减1，再对根节点执行下沉调整。
5. 重复上述过程，直到堆中至多剩一个元素。每轮固定一个最大元素，将剩余排序问题的规模减1，体现减治思想。

### 伪代码

约定下标从0开始，所有循环范围包含端点；范围为空时不执行。n/2 表示整数除法；SiftDown 中的 n 是当前堆的元素个数，不包含末尾已排序部分。

```text
SiftDown(A, n, root):
    重复:
        largest ← root
        left ← 2 * root + 1
        right ← 2 * root + 2

        如果 left < n 且 A[left] > A[largest]:
            largest ← left
        如果 right < n 且 A[right] > A[largest]:
            largest ← right

        如果 largest == root:
            结束重复

        交换 A[root] 和 A[largest]
        root ← largest

HeapSort(A, n):
    对 root 从 n/2 - 1 递减到0:
        SiftDown(A, n, root)

    对 end 从 n - 1 递减到1:
        交换 A[0] 和 A[end]
        SiftDown(A, end, 0)
```

### 代码实现

下沉和排序函数。这里的 n 表示元素个数，而非最后一个元素的下标。


In [14]:
/**
堆排序的下沉操作
@param arr: 待调整的数组
@param n: 当前堆的元素个数（不包含已排序的部分）
@param root: 当前需要下沉的节点索引
*/
void sift_down(int arr[], int n, int root){
    while(1){
        int largest = root;
        int left = 2 * root + 1; // 左子节点索引
        int right = 2 * root + 2; // 右子节点索引

        // 比较当前节点和左孩子
        if(left < n && arr[left] > arr[largest]){
            largest = left;
        }

        // 比较当前节点和右孩子，找到三者中最大的下标
        if(right < n && arr[right] > arr[largest]){
            largest = right;
        }

        // 当前节点已经不小于两个孩子，结束调整
        if(largest == root){
            break;
        }

        int tmp = arr[root];
        arr[root] = arr[largest];
        arr[largest] = tmp;

        root = largest; // 继续向下调整
    }
}

In [15]:
// 堆排序
void heap_sort(int arr[], int n) {
    // 第一阶段：从下往上建立大顶堆
    for (int root = n / 2 - 1; root >= 0; root--) {
        sift_down(arr, n, root);
    }

    // 第二阶段：将当前最大值放到末尾，再恢复剩余部分的堆
    for (int end = n - 1; end > 0; end--) {
        int tmp = arr[0];
        arr[0] = arr[end];
        arr[end] = tmp;

        // 下标 end 及之后已经排好，只调整前 end 个元素
        sift_down(arr, end, 0);
    }
}

### 测试用例


In [16]:
// 逐项比较预期结果，同时输出排序前后的序列。
void test_heap_sort(void) {
    struct HeapCase {
        const char *name;
        int input[8];
        int expected[8];
        int n;
    };
    const struct HeapCase cases[] = {
        {"普通乱序", {29, 10, 14, 37, 13}, {10, 13, 14, 29, 37}, 5},
        {"已经有序", {1, 2, 3, 4, 5}, {1, 2, 3, 4, 5}, 5},
        {"完全逆序", {5, 4, 3, 2, 1}, {1, 2, 3, 4, 5}, 5},
        {"重复值和负数", {3, -1, 3, 0, -1}, {-1, -1, 0, 3, 3}, 5},
        {"全部相等", {7, 7, 7, 7}, {7, 7, 7, 7}, 4},
        {"单个元素", {7}, {7}, 1},
        {"空区间", {0}, {0}, 0}
    };
    int total = (int)(sizeof cases / sizeof cases[0]);
    int passed = 0;

    for (int c = 0; c < total; c++) {
        int arr[8];
        int n = cases[c].n;
        printf("\n%s\n排序前：[", cases[c].name);
        for (int i = 0; i < n; i++) {
            arr[i] = cases[c].input[i];
            printf("%s%d", i == 0 ? "" : ", ", arr[i]);
        }
        printf("]\n");

        heap_sort(arr, n);

        int correct = 1;
        printf("排序后：[");
        for (int i = 0; i < n; i++) {
            printf("%s%d", i == 0 ? "" : ", ", arr[i]);
            if (arr[i] != cases[c].expected[i]) {
                correct = 0;
            }
        }
        printf("]\n预期值：[");
        for (int i = 0; i < n; i++) {
            printf("%s%d", i == 0 ? "" : ", ", cases[c].expected[i]);
        }
        printf("]\n检查结果：%s\n", correct ? "通过" : "失败");
        passed += correct;
    }
    printf("\n堆排序测试：%d/%d组通过。\n", passed, total);
}

// 在Jupyter中显式调用。
{
    test_heap_sort();
}



普通乱序
排序前：[29, 10, 14, 37, 13]
排序后：[10, 13, 14, 29, 37]
预期值：[10, 13, 14, 29, 37]
检查结果：通过

已经有序
排序前：[1, 2, 3, 4, 5]
排序后：[1, 2, 3, 4, 5]
预期值：[1, 2, 3, 4, 5]
检查结果：通过

完全逆序
排序前：[5, 4, 3, 2, 1]
排序后：[1, 2, 3, 4, 5]
预期值：[1, 2, 3, 4, 5]
检查结果：通过

重复值和负数
排序前：[3, -1, 3, 0, -1]
排序后：[-1, -1, 0, 3, 3]
预期值：[-1, -1, 0, 3, 3]
检查结果：通过

全部相等
排序前：[7, 7, 7, 7]
排序后：[7, 7, 7, 7]
预期值：[7, 7, 7, 7]
检查结果：通过

单个元素
排序前：[7]
排序后：[7]
预期值：[7]
检查结果：通过

空区间
排序前：[]
排序后：[]
预期值：[]
检查结果：通过

堆排序测试：7/7组通过。


### 复杂度分析

- **下沉调整**：每次沿树向下一层，最坏时间为 O(log n)；若当前节点已满足堆性质，可立即结束。
- **建堆**：自底向上建堆时，大部分节点靠近叶子，需要调整的层数较少。各层工作量求和为 O(n)，不是简单地将每个节点都按最坏下沉高度计算。
- **完整排序**：排序阶段执行 n-1 轮，每轮交换堆顶并调整剩余堆，最坏总时间为 O(n log n)，平均时间也为 O(n log n)。本实现遇到相等元素不会继续下沉，全部相等的输入可在 O(n) 时间内完成，因此不应将最好时间统一写为 Θ(n log n)。
- **额外空间**：下沉使用循环，只需固定数量的变量，排序算法本身的额外空间为 O(1)，不包含测试数据与预期结果数组。

测试覆盖普通乱序、有序、逆序、负数与重复值、全部相等、单个元素及空区间。测试结果用于验证功能，不用于证明性能优劣。


---

## 题目6：淘汰赛冠军问题

实验目的：

1. 掌握简治法的设计思想；
2. 掌握淘汰赛冠军问题的具体实现过程；
3. 通过这个实例进一步掌握递归技术的运用；
4. 在掌握的基础上编程实现淘汰赛冠军问题的具体实现过程。

实验内容：

假设有 n 个选手进行竞技淘汰赛，最后决出冠军的选手，请设计竞技淘汰比赛的过程，输出结果，输出时要求有文字说明。请任选一种语言编写程序实现上述算法，并分析其算法复杂度。


### 算法思路与伪代码

本实验采用模拟规则：选手编号互不相同，实力较大者获胜；实力相同时，编号较小者获胜。

将当前选手区间分为左右两组，递归决出各组冠军，再让两名组冠军比赛，返回获胜者在原数组中的下标。只有一名选手时直接返回，不安排比赛。人数不必为2的幂，拆分后的两组人数可以相差1。

比赛按递归执行顺序输出：先完成左组，再完成右组，最后输出两组冠军之间的比赛，不按全体选手的轮次交错输出。

```text
Champion(players, left, right):
    // 前提：区间非空，且下标有效
    如果 left == right:
        返回 left

    mid ← left + (right - left) / 2 的整数商
    leftWinner ← Champion(players, left, mid)
    rightWinner ← Champion(players, mid + 1, right)

    如果 players[leftWinner].strength > players[rightWinner].strength:
        winner ← leftWinner
    否则如果 players[leftWinner].strength < players[rightWinner].strength:
        winner ← rightWinner
    否则:
        如果 players[leftWinner].id < players[rightWinner].id:
            winner ← leftWinner
        否则:
            winner ← rightWinner

    输出比赛双方的编号、实力和获胜者编号
    返回 winner

主过程:
    准备选手数组 players 和人数 n
    如果 n <= 0:
        输出“没有参赛选手，无法决出冠军”
    否则:
        winner ← Champion(players, 0, n - 1)
        输出 players[winner] 的编号和实力
```

### 代码实现


In [17]:
typedef struct {
    int id;        // 选手编号，互不相同
    int strength;  // 实力值
} Player;

In [18]:
// 决出闭区间[left, right]中的冠军，返回其数组下标。
// 调用时要求区间非空，且下标有效；选手编号互不相同。
int champion(Player players[], int left, int right) {
    if (left == right) {
        return left;
    }

    int mid = left + (right - left) / 2;

    // 先分别决出左右两组的冠军。
    int leftWinner = champion(players, left, mid);
    int rightWinner = champion(players, mid + 1, right);

    int winner;
    if (players[leftWinner].strength > players[rightWinner].strength) {
        winner = leftWinner;
    } else if (players[leftWinner].strength < players[rightWinner].strength) {
        winner = rightWinner;
    } else {
        // 实力相同，编号较小者获胜；winner保存数组下标。
        if (players[leftWinner].id < players[rightWinner].id) {
            winner = leftWinner;
        } else {
            winner = rightWinner;
        }
    }

    printf("选手%d（实力%d）对阵选手%d（实力%d），选手%d获胜。\n",
           players[leftWinner].id,
           players[leftWinner].strength,
           players[rightWinner].id,
           players[rightWinner].strength,
           players[winner].id);

    return winner;
}


### 测试用例


In [19]:
// expectedIndex为预期冠军下标，空输入使用-1。
int check_champion_case(const char *name, Player players[], int n,
                         int expectedIndex) {
    printf("\n=== %s ===\n", name);
    if (n <= 0) {
        // champion要求非空区间，所以在调用前处理空输入。
        printf("没有参赛选手，无法决出冠军。\n");
        int correct = expectedIndex == -1;
        printf("检查结果：%s\n", correct ? "通过" : "失败");
        return correct;
    }

    printf("参赛人数：%d\n", n);
    int winner = champion(players, 0, n - 1);
    printf("总冠军：选手%d，实力%d，数组下标%d。\n",
           players[winner].id, players[winner].strength, winner);

    int correct = winner == expectedIndex;
    printf("预期冠军下标%d，检查结果：%s\n",
           expectedIndex, correct ? "通过" : "失败");
    return correct;
}

int test_champion(void) {
    int passed = 0;

    // 对应此前A～H的例子，编号1～8；冠军为下标3的选手4。
    Player players1[] = {
        {1, 8}, {2, 3}, {3, 6}, {4, 9},
        {5, 2}, {6, 7}, {7, 5}, {8, 4}
    };
    passed += check_champion_case("8名选手", players1, 8, 3);

    // 人数不是2的幂；同时验证实力相等时按编号决定胜负。
    Player players2[] = {{8, 6}, {3, 6}, {5, 9}, {2, 9}, {7, 4}};
    passed += check_champion_case("5名选手与相同实力", players2, 5, 3);

    Player players3[] = {{9, 5}};
    passed += check_champion_case("仅1名选手，无需比赛", players3, 1, 0);

    // 编号不等于数组下标，编号较小者在右侧。
    Player players4[] = {{8, 6}, {3, 6}};
    passed += check_champion_case("2名选手实力相同", players4, 2, 1);

    passed += check_champion_case("空输入", NULL, 0, -1);

    printf("\n淘汰赛测试：%d/5组通过。\n", passed);
    return 5 - passed;
}

// 在Jupyter中显式调用测试入口。
{
    test_champion();
}



=== 8名选手 ===
参赛人数：8
选手1（实力8）对阵选手2（实力3），选手1获胜。
选手3（实力6）对阵选手4（实力9），选手4获胜。
选手1（实力8）对阵选手4（实力9），选手4获胜。
选手5（实力2）对阵选手6（实力7），选手6获胜。
选手7（实力5）对阵选手8（实力4），选手7获胜。
选手6（实力7）对阵选手7（实力5），选手6获胜。
选手4（实力9）对阵选手6（实力7），选手4获胜。
总冠军：选手4，实力9，数组下标3。
预期冠军下标3，检查结果：通过

=== 5名选手与相同实力 ===
参赛人数：5
选手8（实力6）对阵选手3（实力6），选手3获胜。
选手3（实力6）对阵选手5（实力9），选手5获胜。
选手2（实力9）对阵选手7（实力4），选手2获胜。
选手5（实力9）对阵选手2（实力9），选手2获胜。
总冠军：选手2，实力9，数组下标3。
预期冠军下标3，检查结果：通过

=== 仅1名选手，无需比赛 ===
参赛人数：1
总冠军：选手9，实力5，数组下标0。
预期冠军下标0，检查结果：通过

=== 2名选手实力相同 ===
参赛人数：2
选手8（实力6）对阵选手3（实力6），选手3获胜。
总冠军：选手3，实力6，数组下标1。
预期冠军下标1，检查结果：通过

=== 空输入 ===
没有参赛选手，无法决出冠军。
检查结果：通过

淘汰赛测试：5/5组通过。


### 复杂度分析

设参赛选手人数为 n。以下分析针对一次决出冠军的过程，以一次整数比较、下标运算或固定长度的比赛信息输出为常数时间操作。

#### 1. 时间复杂度

当 n = 1 时，直接返回唯一选手的下标，不安排比赛，时间为 Θ(1)。当 n > 1 时，将选手分为人数近似相等的两组，分别递归求出组冠军，再进行一场比赛。合并时只需常数次实力或编号比较，因此递推关系为：

$$
T(n) = T(\lceil n/2 \rceil) + T(\lfloor n/2 \rfloor) + \Theta(1)
$$

当 n 为2的幂时，可写为 T(n) = 2T(n/2) + Θ(1)，解得 T(n) = Θ(n)。人数不是2的幂时，结论相同。

也可以直接统计比赛场数：每场比赛淘汰一名选手，从 n 名选手中留下唯一冠军，恰好需要 n−1 场比赛。例如，8名选手需要7场，5名选手需要4场，1名选手无需比赛。递归树中有 n 个代表单名选手的叶节点，以及 n−1 个安排比赛的内部节点，总调用次数为 2n−1，因此总时间为 Θ(n)。

当前实现总会递归处理左右两组，不会因某位选手实力特别高而提前结束。实力相等时按编号决定胜负，也只增加常数次比较，所以最好、平均和最坏时间复杂度均为 Θ(n)，通常也记为 O(n)。

每场比赛输出一条记录，共 n−1 条；在上述固定长度输出的计时模型下，包含比赛过程输出的整体时间复杂度仍为 Θ(n)。

#### 2. 额外空间复杂度

每次递归将区间规模缩小约一半，最大递归深度为 O(log n)。每层只保存 mid、leftWinner、rightWinner、winner 等固定数量的局部变量，不复制选手数组，也不保存完整比赛树，因此额外空间复杂度为 O(log n)。

左右两次递归依次执行，左组调用返回后才处理右组；计算峰值空间时，应统计同时存在的调用栈，不能将整棵递归树的所有调用次数相加。

选手数组本身占用 O(n) 空间。若将输入数组与递归栈一并计入，总空间为 O(n)；上述 O(log n) 指不包含输入数组的算法额外空间，也不包含笔记本界面保存输出文本所占的空间。

#### 3. 边界情况

- n = 1：时间和额外空间均为 O(1)，直接返回该选手。
- n = 0：由调用方直接输出“没有参赛选手，无法决出冠军”，不调用要求非空区间的 champion，时间和额外空间均为 O(1)。


---

## 题目7：邮局选址问题（选做）

实验目的：

强化学生利用分治算法求解实际问题的能力。

实验内容：

在一个按照东西和南北方向划分成规整街区的城市里，n 个居民点散乱地分布在不同的街区中。用 x 坐标表示东西向，用 y 坐标表示南北向。各居民点的位置可以由坐标 (x,y) 来表示。

街区中任意两点 (x1,y1) 和 (x2,y2) 之间的距离可以用数值 |x1-x2|+|y1-y2| 来度量。居民们希望在城市中选择建立邮局的最佳位置，使 n 个居民点到邮局的距离总和最小。


### 算法思路

邮局位置记为 (a, b)，总距离为 Σ|xᵢ-a| + Σ|yᵢ-b|。两个求和分别只依赖横坐标和纵坐标，因此可以独立选择最优的 a、b，再组合成邮局位置；该位置不必是已有居民点。

一维绝对距离和在中位数处取得最小值：将排序后的两端坐标配对，每一对到区间内任意位置的距离和均达到最小；所有配对区间的交集为奇数时的中间坐标，或偶数时中间两个坐标之间的区间。因此，本实现统一选择排序后下标 k=(n-1)/2 处的坐标，偶数时选择较小的中位数。

1. 将居民点的横、纵坐标分别复制到数组 X、Y，保留原居民点数据。
2. 分别使用快速选择求 X、Y 的第 k 个下标对应的元素值，得到 a、b。
3. 快速选择使用本题独立定义的划分函数：划分得到基准下标 p，若 k=p 则返回基准值；否则只递归进入包含目标下标的一侧，无需将整个数组排序。
4. 使用原居民点坐标累计 |xᵢ-a|+|yᵢ-b|，输出一个最优位置及最小距离总和。
5. 释放临时坐标数组。没有居民点时输出说明并结束。

### 伪代码实现

约定：数组下标从0开始，区间包含两端；k 始终是原数组中的目标下标。Partition 对应本题的 post_partition，返回基准最终下标；QuickSelect 返回元素值。距离总和使用足够宽的整数类型保存。

```text
QuickSelect(A, left, right, k):
    // 前提：left <= k <= right，且下标有效
    如果 left == right:
        返回 A[left]

    p ← Partition(A, left, right)

    如果 k == p:
        返回 A[p]
    否则如果 k < p:
        返回 QuickSelect(A, left, p - 1, k)
    否则:
        返回 QuickSelect(A, p + 1, right, k)


PostOffice(points, n):
    如果 n <= 0:
        输出“没有居民点，无法确定邮局位置”
        返回

    申请长度为 n 的整数数组 X、Y
    如果任一申请失败:
        释放已申请的空间
        输出内存分配失败说明
        返回

    对 i 从 0 到 n - 1（包含端点）:
        X[i] ← points[i].x
        Y[i] ← points[i].y

    k ← (n - 1) / 2 的整数商
    a ← QuickSelect(X, 0, n - 1, k)
    b ← QuickSelect(Y, 0, n - 1, k)

    total ← 0
    对 i 从 0 到 n - 1（包含端点）:
        dx ← 转为宽整数的 points[i].x - a
        dy ← 转为宽整数的 points[i].y - b
        total ← total + |dx| + |dy|

    输出“最优邮局位置：(a, b)”
    输出“最小距离总和：total”
    释放 X、Y
```


### 代码实现

独立运行本题时，按下方顺序执行：头文件与 Point 定义 → 本题划分函数与 quick_select → post_office → 测试。无需运行第二题。

In [20]:
typedef struct {
    int x;
    int y;
} Point;


In [ ]:
// 返回基准的最终下标；左侧小于基准，右侧大于或等于基准。
int post_partition(int arr[], int left, int right) {
    int pivot = arr[right];
    int i = left;

    for (int j = left; j < right; j++) {
        if (arr[j] < pivot) {
            int tmp = arr[i];
            arr[i] = arr[j];
            arr[j] = tmp;
            i++;
        }
    }

    int tmp = arr[i];
    arr[i] = arr[right];
    arr[right] = tmp;
    return i;
}

// 返回排序后下标k处的元素值，会改变数组顺序。
// 要求0 <= left <= k <= right，且数组下标有效。
int quick_select(int arr[], int left, int right, int k) {
    if (left == right) {
        return arr[left];
    }

    int p = post_partition(arr, left, right);

    if (k == p) {
        return arr[p];
    } else if (k < p) {
        return quick_select(arr, left, p - 1, k);
    } else {
        return quick_select(arr, p + 1, right, k);
    }
}


In [22]:
// 根据居民点坐标，计算并输出一个最优邮局位置。
// 要求points包含n个有效居民点。
void post_office(Point points[], int n) {
    if (n <= 0) {
        printf("没有居民点，无法确定邮局位置。\n");
        return;
    }

    // 分别保存横、纵坐标，避免快速选择改变原居民点数据。
    int *X = malloc((size_t)n * sizeof *X);
    int *Y = malloc((size_t)n * sizeof *Y);

    if (X == NULL || Y == NULL) {
        // free(NULL)也是合法的，可释放已经成功申请的那一部分。
        free(X);
        free(Y);
        fprintf(stderr, "坐标数组内存分配失败。\n");
        return;
    }

    for (int i = 0; i < n; i++) {
        X[i] = points[i].x;
        Y[i] = points[i].y;
    }

    // 偶数个元素时，选择中间两个坐标中较小的一个。
    int k = (n - 1) / 2;
    int a = quick_select(X, 0, n - 1, k);
    int b = quick_select(Y, 0, n - 1, k);

    // 累计各居民点到邮局(a, b)的曼哈顿距离。

    long long total = 0;

    for (int i = 0; i < n; i++) {
        long long dx = (long long)points[i].x - a;
        long long dy = (long long)points[i].y - b;

        total += llabs(dx) + llabs(dy);
    }

    printf("最优邮局位置：(%d, %d)\n", a, b);
    printf("最小距离总和：%lld\n", total);  

    free(X);
    free(Y);
}

### 测试用例

依次执行函数定义后运行下方单元格，对照每组的预期位置与最小距离总和。偶数个坐标可能有多个最优解，本测试按当前实现固定选择较小中位数。


In [23]:
// 当前post_office直接输出结果，因此每组先给出预期值，便于对照。
// 坐标和总距离均按“选择较小中位数”的约定给出。
void show_post_office_case(const char *name, Point points[], int n,
                           int expectedX, int expectedY,
                           long long expectedTotal) {
    printf("\n=== %s ===\n", name);
    printf("居民点：");
    for (int i = 0; i < n; i++) {
        printf("(%d, %d) ", points[i].x, points[i].y);
    }
    printf("\n预期位置：(%d, %d)，预期最小距离总和：%lld\n",
           expectedX, expectedY, expectedTotal);
    printf("实际输出：\n");
    post_office(points, n);
}

void test_post_office(void) {
    Point odd[] = {{1, 2}, {2, 8}, {8, 3}};
    show_post_office_case("奇数个居民点，邮局不在居民点上",
                          odd, 3, 2, 3, 13);

    Point even[] = {{1, 1}, {3, 7}, {7, 3}, {9, 9}};
    show_post_office_case("偶数个居民点，选择较小中位数",
                          even, 4, 3, 3, 24);

    Point negative[] = {{-4, -2}, {0, 4}, {6, 1}};
    show_post_office_case("包含负坐标",
                          negative, 3, 0, 1, 16);

    Point repeated[] = {{2, 3}, {2, 3}, {2, 3}, {8, 9}};
    show_post_office_case("重复居民点",
                          repeated, 4, 2, 3, 12);

    Point single[] = {{-2, 5}};
    show_post_office_case("只有一个居民点",
                          single, 1, -2, 5, 0);

    Point horizontal[] = {{1, 5}, {9, 5}};
    show_post_office_case("两个居民点位于同一水平线",
                          horizontal, 2, 1, 5, 8);

    printf("\n=== 空输入 ===\n");
    printf("预期：没有居民点，无法确定邮局位置。\n实际输出：\n");
    post_office(NULL, 0);
}

// 在Jupyter中显式调用；先按顺序执行本题上方的三个定义单元格。
{
    test_post_office();
}



=== 奇数个居民点，邮局不在居民点上 ===
居民点：(1, 2) (2, 8) (8, 3) 
预期位置：(2, 3)，预期最小距离总和：13
实际输出：
最优邮局位置：(2, 3)
最小距离总和：13

=== 偶数个居民点，选择较小中位数 ===
居民点：(1, 1) (3, 7) (7, 3) (9, 9) 
预期位置：(3, 3)，预期最小距离总和：24
实际输出：
最优邮局位置：(3, 3)
最小距离总和：24

=== 包含负坐标 ===
居民点：(-4, -2) (0, 4) (6, 1) 
预期位置：(0, 1)，预期最小距离总和：16
实际输出：
最优邮局位置：(0, 1)
最小距离总和：16

=== 重复居民点 ===
居民点：(2, 3) (2, 3) (2, 3) (8, 9) 
预期位置：(2, 3)，预期最小距离总和：12
实际输出：
最优邮局位置：(2, 3)
最小距离总和：12

=== 只有一个居民点 ===
居民点：(-2, 5) 
预期位置：(-2, 5)，预期最小距离总和：0
实际输出：
最优邮局位置：(-2, 5)
最小距离总和：0

=== 两个居民点位于同一水平线 ===
居民点：(1, 5) (9, 5) 
预期位置：(1, 5)，预期最小距离总和：8
实际输出：
最优邮局位置：(1, 5)
最小距离总和：8

=== 空输入 ===
预期：没有居民点，无法确定邮局位置。
实际输出：
没有居民点，无法确定邮局位置。
